In [1]:
# explore the cells of filtered_records_on_zeolite.json starting with how many records there are
import json

with open('filtered_records_on_zeolite.json', 'r') as f:
    data = json.load(f)

len(data)

42678

In [2]:
#also check what fields are there maybe by print a record
print(data[0].keys())

dict_keys(['_id', 'doi', 'file_loc', 'file_type', 'title', 'abstract', 'modified', 'operations', 'materials', 'paragraphs', 'connections', 'meta_tags', 'synth_params', 'tables', 'keywords', 'para_class_type', 'token_class_type', 'title_targets', 'title_metas'])


In [3]:
#print a full record to see what publisher names look like
print(data[78])

{'_id': '6369a88518b1bdb7c36e47ad', 'doi': '10.1002/er.3219', 'file_loc': '/data/synthesis-project/files/htmls/101002er3219.html', 'file_type': 'html', 'title': 'Direct conversion of syngas to DME over CuO–ZnO–Al2O3/HZSM‐5 nanocatalyst synthesized via ultrasound‐assisted co‐precipitation method: New insights into the role of gas injection', 'abstract': 'In this study, direct synthesis of dimethyl ether (DME) is conducted over a bifunctional CuO-ZnO-Al2O3/H Zeolite Socony Mobil-5 (HZSM-5) nanocatalyst. A hybrid method of ultrasound-assisted co-precipitation is used for the synthesis of catalysts, and the effect of gas injection during sonication is investigated. The physicochemical characteristics of the catalysts are analysed by X-ray diffraction (XRD), field emission scanning electron microscopy (FESEM), particle size distribution (PSD), energy dispersive X-ray (EDX), Brunauer-Emmett-Teller (BET) and Fourier-transformed infrared (FTIR) methods. In the absence of gas injection, the ace

In [4]:
# Check if there are any publisher-related fields
# Let's look for fields that might contain publisher info
print("All keys in the record:")
for key in data[0].keys():
    print(f"  {key}")

print("\n" + "="*50)

# Check if any field names contain 'publish', 'journal', 'source', etc.
publisher_related_keys = []
for key in data[0].keys():
    if any(word in key.lower() for word in ['publish', 'journal', 'source', 'venue', 'issn']):
        publisher_related_keys.append(key)

if publisher_related_keys:
    print(f"Found potential publisher-related fields: {publisher_related_keys}")
    for key in publisher_related_keys:
        print(f"{key}: {data[0][key]}")
else:
    print("No obvious publisher-related fields found in the main record structure")

print("\n" + "="*50)

# Let's also check the DOI - sometimes we can infer publisher from DOI
doi = data[0].get('doi', '')
print(f"DOI: {doi}")

# Common publisher DOI prefixes
publisher_prefixes = {
    '10.1002': 'Wiley',
    '10.1016': 'Elsevier', 
    '10.1021': 'American Chemical Society',
    '10.1038': 'Nature Publishing Group',
    '10.1007': 'Springer',
    '10.1039': 'Royal Society of Chemistry',
    '10.1103': 'American Physical Society',
    '10.1088': 'IOP Publishing'
}

if doi:
    doi_prefix = '/'.join(doi.split('/')[:2])
    if doi_prefix in publisher_prefixes:
        print(f"Based on DOI prefix {doi_prefix}, this appears to be published by: {publisher_prefixes[doi_prefix]}")
    else:
        print(f"DOI prefix {doi_prefix} not in common publisher list")

All keys in the record:
  _id
  doi
  file_loc
  file_type
  title
  abstract
  modified
  operations
  materials
  paragraphs
  connections
  meta_tags
  synth_params
  tables
  keywords
  para_class_type
  token_class_type
  title_targets
  title_metas

No obvious publisher-related fields found in the main record structure

DOI: 10.1002/1521-4079(200111)36:11<1197::aid-crat1197>3.0.co;2-d
DOI prefix 10.1002/1521-4079(200111)36:11<1197::aid-crat1197>3.0.co;2-d not in common publisher list


In [5]:
# Analyze publisher information for ALL records
from collections import Counter

# Extended publisher DOI prefixes
publisher_prefixes = {
    '10.1002': 'Wiley',
    '10.1016': 'Elsevier', 
    '10.1021': 'American Chemical Society (ACS)',
    '10.1038': 'Nature Publishing Group',
    '10.1007': 'Springer',
    '10.1039': 'Royal Society of Chemistry (RSC)',
    '10.1103': 'American Physical Society (APS)',
    '10.1088': 'IOP Publishing',
    '10.1063': 'AIP Publishing',
    '10.1080': 'Taylor & Francis',
    '10.1126': 'Science (AAAS)',
    '10.1149': 'The Electrochemical Society',
    '10.1021': 'American Chemical Society',
    '10.1006': 'Academic Press',
    '10.1023': 'Springer (Kluwer)',
    '10.1081': 'Taylor & Francis',
    '10.1155': 'Hindawi',
    '10.3390': 'MDPI',
    '10.1371': 'PLOS',
    '10.1186': 'BioMed Central',
    '10.1038': 'Nature',
    '10.1073': 'PNAS'
}

print(f"Analyzing {len(data)} records for publisher information...")
print("="*60)

# Track publishers and DOI patterns
publisher_counts = Counter()
doi_prefixes = Counter()
records_with_doi = 0
records_without_doi = 0
unknown_publishers = Counter()

# Sample records for each publisher
publisher_samples = {}

for i, record in enumerate(data):
    doi = record.get('doi', '')
    
    if doi:
        records_with_doi += 1
        # Extract DOI prefix (first two parts)
        try:
            doi_prefix = '/'.join(doi.split('/')[:2])
            doi_prefixes[doi_prefix] += 1
            
            if doi_prefix in publisher_prefixes:
                publisher = publisher_prefixes[doi_prefix]
                publisher_counts[publisher] += 1
                
                # Store sample record for each publisher (first occurrence)
                if publisher not in publisher_samples:
                    publisher_samples[publisher] = {
                        'index': i,
                        'doi': doi,
                        'title': record.get('title', 'No title')
                    }
            else:
                unknown_publishers[doi_prefix] += 1
        except:
            unknown_publishers['Invalid DOI format'] += 1
    else:
        records_without_doi += 1

print(f"Records with DOI: {records_with_doi}")
print(f"Records without DOI: {records_without_doi}")
print(f"Total records: {len(data)}")
print()

print("PUBLISHER DISTRIBUTION:")
print("="*40)
for publisher, count in publisher_counts.most_common():
    percentage = (count / len(data)) * 100
    print(f"{publisher}: {count} records ({percentage:.1f}%)")

print()
print("UNKNOWN DOI PREFIXES (Top 10):")
print("="*40)
for prefix, count in unknown_publishers.most_common(10):
    percentage = (count / len(data)) * 100
    print(f"{prefix}: {count} records ({percentage:.1f}%)")

print()
print("ALL DOI PREFIXES (Top 20):")
print("="*40)
for prefix, count in doi_prefixes.most_common(20):
    publisher = publisher_prefixes.get(prefix, 'Unknown')
    percentage = (count / len(data)) * 100
    print(f"{prefix}: {count} records ({percentage:.1f}%) - {publisher}")

Analyzing 42678 records for publisher information...
Records with DOI: 42678
Records without DOI: 0
Total records: 42678

PUBLISHER DISTRIBUTION:

UNKNOWN DOI PREFIXES (Top 10):
10.1088/0953-8984: 18 records (0.0%)
10.1088/2053-1591: 15 records (0.0%)
10.1088/0957-4484: 11 records (0.0%)
10.1016/s0926-860x(03)00636-7: 9 records (0.0%)
10.1016/s0926-860x(01)00932-2: 8 records (0.0%)
10.1140/epjst: 8 records (0.0%)
10.1016/s1387-1811(03)00360-3: 7 records (0.0%)
10.1016/s0008-8846(01)00508-7: 7 records (0.0%)
10.1016/s0926-860x(03)00284-9: 7 records (0.0%)
10.1016/s0926-860x(00)00769-9: 7 records (0.0%)

ALL DOI PREFIXES (Top 20):
10.1088/0953-8984: 18 records (0.0%) - Unknown
10.1088/2053-1591: 15 records (0.0%) - Unknown
10.1088/0957-4484: 11 records (0.0%) - Unknown
10.1016/s0926-860x(03)00636-7: 9 records (0.0%) - Unknown
10.1016/s0926-860x(01)00932-2: 8 records (0.0%) - Unknown
10.1140/epjst: 8 records (0.0%) - Unknown
10.1016/s1387-1811(03)00360-3: 7 records (0.0%) - Unknown
10.101

In [7]:
# Let's first check some sample DOIs to understand the format
print("SAMPLE DOIs (first 10 records):")
for i in range(min(10, len(data))):
    doi = data[i].get('doi', 'No DOI')
    print(f"Record {i}: {doi}")

print("\n" + "="*60)

# The issue is that DOIs in the format like "10.1088/0953-8984" should map to "10.1088"
# But our parsing was taking the full path. Let's fix this:

print("FIXING DOI PARSING...")
publisher_prefixes = {
    '10.1002': 'Wiley',  
    '10.1016': 'Elsevier',  
    '10.1021': 'American Chemical Society (ACS)',  
    '10.1038': 'Nature Publishing Group',  
    '10.1007': 'Springer',  
    '10.1039': 'Royal Society of Chemistry (RSC)',  
    '10.1103': 'American Physical Society (APS)',  
    '10.1088': 'IOP Publishing',  
    '10.1063': 'AIP Publishing',  
    '10.1080': 'Taylor & Francis (Informa UK Limited)',  
    '10.1126': 'Science (AAAS)',  
    '10.1149': 'The Electrochemical Society',  
    '10.1006': 'Academic Press',  
    '10.1023': 'Springer (Kluwer)',  
    '10.1081': 'Taylor & Francis / Marcel Dekker imprint',   
    '10.1155': 'Hindawi',  
    '10.3390': 'MDPI',  
    '10.1371': 'PLOS',  
    '10.1186': 'BioMed Central',  
    '10.1073': 'PNAS',  
    '10.1140': 'Springer',
    '10.1111': 'Wiley / Wiley-Blackwell',
    '10.1107': 'International Union of Crystallography (IUCr)', 
    '10.1617': 'Springer / RILEM Publishing'
}

# Re-analyze with correct DOI parsing
publisher_counts = Counter()
doi_prefixes = Counter()
unknown_publishers = Counter()
publisher_samples = {}

for i, record in enumerate(data):
    doi = record.get('doi', '')
    
    if doi:
        # Extract just the publisher prefix (e.g., "10.1016" from "10.1016/j.jcat.2003.05.001")
        try:
            # Split by '/' and take the first part after '10.'
            parts = doi.split('/')
            if len(parts) >= 1 and parts[0].startswith('10.'):
                doi_prefix = parts[0]  # This gets us "10.1016", "10.1088", etc.
                doi_prefixes[doi_prefix] += 1
                
                if doi_prefix in publisher_prefixes:
                    publisher = publisher_prefixes[doi_prefix]
                    publisher_counts[publisher] += 1
                    
                    # Store sample record for each publisher (first occurrence)
                    if publisher not in publisher_samples:
                        publisher_samples[publisher] = {
                            'index': i,
                            'doi': doi,
                            'title': record.get('title', 'No title')
                        }
                else:
                    unknown_publishers[doi_prefix] += 1
        except:
            unknown_publishers['Invalid DOI format'] += 1

print(f"\nCORRECTED PUBLISHER DISTRIBUTION:")
print("="*40)
for publisher, count in publisher_counts.most_common():
    percentage = (count / len(data)) * 100
    print(f"{publisher}: {count} records ({percentage:.1f}%)")

print(f"\nUNKNOWN DOI PREFIXES (Top 10):")
print("="*40)
for prefix, count in unknown_publishers.most_common(10):
    percentage = (count / len(data)) * 100
    print(f"{prefix}: {count} records ({percentage:.1f}%)")

print(f"\nTOP DOI PREFIXES:")
print("="*40)
for prefix, count in doi_prefixes.most_common(30):
    publisher = publisher_prefixes.get(prefix, 'Unknown')
    percentage = (count / len(data)) * 100
    print(f"{prefix}: {count} records ({percentage:.1f}%) - {publisher}")

SAMPLE DOIs (first 10 records):
Record 0: 10.1002/1521-4079(200111)36:11<1197::aid-crat1197>3.0.co;2-d
Record 1: 10.1006/jcat.1994.1275
Record 2: 10.1016/j.apcata.2004.06.029
Record 3: 10.1016/j.apcata.2005.04.047
Record 4: 10.1016/j.apcatb.2014.11.047
Record 5: 10.1016/j.cattod.2013.10.061
Record 6: 10.1016/j.cattod.2012.09.002
Record 7: 10.1006/jcat.1994.1318
Record 8: 10.1016/j.egypro.2017.03.1375
Record 9: 10.1002/admi.202000348

FIXING DOI PARSING...

CORRECTED PUBLISHER DISTRIBUTION:
Elsevier: 24978 records (58.5%)
American Chemical Society (ACS): 5671 records (13.3%)
Royal Society of Chemistry (RSC): 4283 records (10.0%)
Wiley: 3322 records (7.8%)
Springer: 3160 records (7.4%)
Academic Press: 536 records (1.3%)
Nature Publishing Group: 258 records (0.6%)
Springer (Kluwer): 155 records (0.4%)
Wiley / Wiley-Blackwell: 145 records (0.3%)
American Physical Society (APS): 65 records (0.2%)
IOP Publishing: 48 records (0.1%)
The Electrochemical Society: 24 records (0.1%)
BioMed Central

In [ ]:
# Create a list of Wiley papers (DOI prefixes 10.1111 and 10.1002)
wiley_papers = []

for record in data:
    doi = record.get('doi', '')
    if doi:
        doi_prefix = doi.split('/')[0] if '/' in doi else doi
        if doi_prefix in ['10.1111', '10.1002']:
            wiley_papers.append({
                'doi': doi,
                'title': record.get('title', 'No title')
            })

# Save to JSON file
with open('wiley_papers.json', 'w') as f:
    json.dump(wiley_papers, f, indent=2)

print(f"Total Wiley papers found: {len(wiley_papers)}")
print(f"JSON file 'wiley_papers.json' created successfully")